In [0]:
df_raw = spark.read.csv(
    "/Volumes/workspace/default/bronze/amazon.csv",
    header=True,
    inferSchema=False
)

print(f"Rows: {df_raw.count()}")
print(f"Columns: {len(df_raw.columns)}")
df_raw.printSchema()

Rows: 1465
Columns: 16
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: string (nullable = true)
 |-- actual_price: string (nullable = true)
 |-- discount_percentage: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- rating_count: string (nullable = true)
 |-- about_product: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- review_title: string (nullable = true)
 |-- review_content: string (nullable = true)
 |-- img_link: string (nullable = true)
 |-- product_link: string (nullable = true)



In [0]:
df_raw.select(
    "product_name",
    "category",
    "discounted_price",
    "actual_price",
    "discount_percentage",
    "rating",
    "rating_count"
).show(5, truncate = False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------+----------------+------------+-------------------+------+------------+
|product_name                                                                                                                                                                                           |category                                                                         |discounted_price|actual_price|discount_percentage|rating|rating_count|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------+---------

Data Cleaning

In [0]:
df_raw.columns

['product_id',
 'product_name',
 'category',
 'discounted_price',
 'actual_price',
 'discount_percentage',
 'rating',
 'rating_count',
 'about_product',
 'user_id',
 'user_name',
 'review_id',
 'review_title',
 'review_content',
 'img_link',
 'product_link']

In [0]:
columns_to_drop = [
    "about_product", "img_link", "product_link", "review_id", "review_title", "review_content", "user_id", "user_name"
]

df_dropped = df_raw.drop(*columns_to_drop)
print(f"Columns remaining: {len(df_dropped.columns)}")
print(df_dropped.columns)

Columns remaining: 8
['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count']


In [0]:
df_dropped.select("discounted_price").show(5, truncate=False)

+----------------+
|discounted_price|
+----------------+
|₹399            |
|₹199            |
|₹199            |
|₹329            |
|₹154            |
+----------------+
only showing top 5 rows


In [0]:
spark.conf.set("spark.sql.ansi.enabled", "false")

In [0]:
from pyspark.sql.functions import regexp_replace, col

df_prices = df_dropped \
    .withColumn("discounted_price",
        regexp_replace(col("discounted_price"), "[^0-9.]", "").cast("float")) \
    .withColumn("actual_price",
        regexp_replace(col("actual_price"), "[^0-9.]", "").cast("float")) \
    .withColumn("discount_percentage",
        regexp_replace(col("discount_percentage"), "[^0-9.]", "").cast("float")) \
    .withColumn("rating",
        regexp_replace(col("rating"), "[^0-9.]", "").cast("float")) \
    .withColumn("rating_count",
        regexp_replace(col("rating_count"), "[^0-9]", "").cast("integer"))

df_prices.select(
    "discounted_price", "actual_price", 
    "discount_percentage", "rating", "rating_count"
).show(5)

+----------------+------------+-------------------+------+------------+
|discounted_price|actual_price|discount_percentage|rating|rating_count|
+----------------+------------+-------------------+------+------------+
|           399.0|      1099.0|               64.0|   4.2|       24269|
|           199.0|       349.0|               43.0|   4.0|       43994|
|           199.0|      1899.0|               90.0|   3.9|        7928|
|           329.0|       699.0|               53.0|   4.2|       94363|
|           154.0|       399.0|               61.0|   4.2|       16905|
+----------------+------------+-------------------+------+------------+
only showing top 5 rows


In [0]:
df_prices.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- discounted_price: float (nullable = true)
 |-- actual_price: float (nullable = true)
 |-- discount_percentage: float (nullable = true)
 |-- rating: float (nullable = true)
 |-- rating_count: integer (nullable = true)



In [0]:
from pyspark.sql.functions import count, when, isnan

for col_name in df_prices.columns:
    null_count = df_prices.filter(
        col(col_name).isNull()
    ).count()
    print(f"{col_name}: {null_count} nulls")

product_id: 0 nulls
product_name: 0 nulls
category: 0 nulls
discounted_price: 19 nulls
actual_price: 22 nulls
discount_percentage: 36 nulls
rating: 23 nulls
rating_count: 23 nulls


In [0]:
df_dropped.filter(
    col("discounted_price").isNull() | 
    (col("discounted_price") == "") |
    col("discounted_price").rlike("^[^0-9]*$")
).select("product_name", "discounted_price", "actual_price", "discount_percentage").show(10, truncate=False)

+--------------------------------------------------------------------------------------------------------------------+----------------------+---------------------------------------------+-------------------------------------------+
|product_name                                                                                                        |discounted_price      |actual_price                                 |discount_percentage                        |
+--------------------------------------------------------------------------------------------------------------------+----------------------+---------------------------------------------+-------------------------------------------+
|"EGate i9 Pro-Max 1080p Native Full HD Projector 4k Support | 3600 L (330 ANSI ) | 150"" (381 cm) Large Screen | VGA| HDMI                 | SD Card                                     | USB                                       |
|"Fire-Boltt Visionary 1.78"" AMOLED Bluetooth Calling Smartwatch with 3

In [0]:
df_dropped.filter(
    col("product_name").contains("EGate i9")
).select("product_name", "discounted_price", "actual_price").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------+----------------+------------+
|product_name                                                                                                        |discounted_price|actual_price|
+--------------------------------------------------------------------------------------------------------------------+----------------+------------+
|"EGate i9 Pro-Max 1080p Native Full HD Projector 4k Support | 3600 L (330 ANSI ) | 150"" (381 cm) Large Screen | VGA| HDMI           | SD Card    |
+--------------------------------------------------------------------------------------------------------------------+----------------+------------+



In [0]:
df_clean = df_prices.dropna(subset=[
    "discounted_price",
    "actual_price", 
    "discount_percentage",
    "rating",
    "rating_count"
])

print(f"Rows before: {df_prices.count()}")
print(f"Rows after: {df_clean.count()}")
print(f"Rows dropped: {df_prices.count() - df_clean.count()}")

Rows before: 1465
Rows after: 1406
Rows dropped: 59


In [0]:
from pyspark.sql.functions import split

df_categories = df_clean \
    .withColumn("main_category", split(col("category"), "\\|").getItem(0)) \
    .withColumn("sub_category",  split(col("category"), "\\|").getItem(1))

df_categories.select("category", "main_category", "sub_category").show(5, truncate=False)

+---------------------------------------------------------------------------------+---------------------+-----------------------+
|category                                                                         |main_category        |sub_category           |
+---------------------------------------------------------------------------------+---------------------+-----------------------+
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|Accessories&Peripherals|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|Accessories&Peripherals|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|Accessories&Peripherals|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Computers&Accessories|Accessories&Peripherals|
|Computers&Accessories|Accessories&Peripherals|Cables&Accessories|Cables|USBCables|Compute

In [0]:
df_silver = df_categories.drop("category")

print(f"Final columns: {df_silver.columns}")
print(f"Final row count: {df_silver.count()}")
df_silver.printSchema()

Final columns: ['product_id', 'product_name', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'main_category', 'sub_category']
Final row count: 1406
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- discounted_price: float (nullable = true)
 |-- actual_price: float (nullable = true)
 |-- discount_percentage: float (nullable = true)
 |-- rating: float (nullable = true)
 |-- rating_count: integer (nullable = true)
 |-- main_category: string (nullable = true)
 |-- sub_category: string (nullable = true)



In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.silver")

DataFrame[]

In [0]:
df_silver.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/silver/amazon_cleaned")

print("Silver layer written successfully")

Silver layer written successfully


df_silver.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/silver/amazon_cleaned_csv")

Gold Layer

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.gold")

DataFrame[]

In [0]:
from pyspark.sql.functions import avg, count, sum, max, round as spark_round

df_gold = df_silver \
    .groupBy("main_category") \
    .agg(
        count("product_id").alias("total_products"),
        spark_round(avg("discounted_price"), 2).alias("avg_discounted_price"),
        spark_round(avg("actual_price"), 2).alias("avg_actual_price"),
        spark_round(avg("discount_percentage"), 2).alias("avg_discount_pct"),
        spark_round(avg("rating"), 2).alias("avg_rating"),
        sum("rating_count").alias("total_reviews")
    ) \
    .orderBy("total_products", ascending=False)

df_gold.show(truncate=False)

+-----------------------------+--------------+--------------------+----------------+----------------+----------+-------------+
|main_category                |total_products|avg_discounted_price|avg_actual_price|avg_discount_pct|avg_rating|total_reviews|
+-----------------------------+--------------+--------------------+----------------+----------------+----------+-------------+
|Electronics                  |476           |6341.61             |10259.7         |48.96           |4.08      |14902561     |
|Home&Kitchen                 |447           |2331.13             |4165.79         |40.17           |4.04      |2990077      |
|Computers&Accessories        |445           |768.04              |1561.34         |53.88           |4.15      |7685363      |
|OfficeProducts               |30            |308.63              |404.6           |11.13           |4.31      |142246       |
|HomeImprovement              |2             |337.0               |799.0           |57.5            |4.25      

In [0]:
df_gold_clean = df_gold \
    .filter(col("avg_rating") <= 5) \
    .filter(col("avg_discount_pct") <= 100)

df_gold_clean.show(truncate=False)

+---------------------+--------------+--------------------+----------------+----------------+----------+-------------+
|main_category        |total_products|avg_discounted_price|avg_actual_price|avg_discount_pct|avg_rating|total_reviews|
+---------------------+--------------+--------------------+----------------+----------------+----------+-------------+
|Electronics          |476           |6341.61             |10259.7         |48.96           |4.08      |14902561     |
|Home&Kitchen         |447           |2331.13             |4165.79         |40.17           |4.04      |2990077      |
|Computers&Accessories|445           |768.04              |1561.34         |53.88           |4.15      |7685363      |
|OfficeProducts       |30            |308.63              |404.6           |11.13           |4.31      |142246       |
|HomeImprovement      |2             |337.0               |799.0           |57.5            |4.25      |8566         |
|MusicalInstruments   |2             |638.0     

In [0]:
from pyspark.sql.functions import regexp_replace

df_gold_clean = df_gold_clean \
    .withColumn("main_category",
        regexp_replace(col("main_category"), "&", " & ")) \
    .withColumn("main_category",
        regexp_replace(col("main_category"), "([a-z])([A-Z])", "$1 $2"))

df_gold_clean.show(truncate=False)

+-----------------------+--------------+--------------------+----------------+----------------+----------+-------------+
|main_category          |total_products|avg_discounted_price|avg_actual_price|avg_discount_pct|avg_rating|total_reviews|
+-----------------------+--------------+--------------------+----------------+----------------+----------+-------------+
|Electronics            |476           |6341.61             |10259.7         |48.96           |4.08      |14902561     |
|Home & Kitchen         |447           |2331.13             |4165.79         |40.17           |4.04      |2990077      |
|Computers & Accessories|445           |768.04              |1561.34         |53.88           |4.15      |7685363      |
|Office Products        |30            |308.63              |404.6           |11.13           |4.31      |142246       |
|Home Improvement       |2             |337.0               |799.0           |57.5            |4.25      |8566         |
|Musical Instruments    |2      

In [0]:
df_gold_clean.write \
    .mode("overwrite") \
    .parquet("/Volumes/workspace/default/gold/category_analytics")

print("Gold layer written successfully")

Gold layer written successfully


In [0]:
df_gold_clean.write \
    .mode("overwrite") \
    .saveAsTable("workspace.default.category_analytics")